In [1]:
import numpy as np
import pandas as pd
import yaml
from linearmodels.panel import PanelOLS

# -----------------------------
# 0) Display / settings
# -----------------------------
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.3f}".format)
with open("../../Settings.yaml", "r") as file:
    Setting = yaml.safe_load(file)

# -----------------------------
# 1) Load data
# -----------------------------
file_path = f"{Setting['Output_Path_Ajusted']}/Adjusted.xlsx"
Dataset = pd.read_excel(file_path, sheet_name="Dataset_for_Model")

# -----------------------------
# 2) Variable construction
# -----------------------------
Dataset["ln_Electricity"] = np.log(Dataset["Electricity"])
Dataset["ln_Electricity_unit_price"] = np.log(Dataset["Electricity_unit_price"])
Dataset["Intensity_Temp"] = Dataset["Elec_boe_intensity"] * Dataset["Temprature"]

# -----------------------------
# 3) Set panel index (entity, time)
# -----------------------------
Dataset = Dataset.set_index(["Industry_Code", "Year"]).sort_index()

# -----------------------------
# 4) First-stage regression (panel FE)
# ln(Electricity_it) on ln(UnitPrice_it) and (ElecIntensity_i × Temperature_t)
# with industry fixed effects and year fixed effects
# -----------------------------
y = Dataset["ln_Electricity"]
X = Dataset[["ln_Electricity_unit_price", "Intensity_Temp"]]

mod1 = PanelOLS(y, X, entity_effects=True, time_effects=True)

# Cluster-robust standard errors at the industry (entity) level
res1 = mod1.fit(cov_type="clustered", cluster_entity=True)

print(res1.summary)


                          PanelOLS Estimation Summary                           
Dep. Variable:         ln_Electricity   R-squared:                        0.6878
Estimator:                   PanelOLS   R-squared (Between):              0.4081
No. Observations:                 504   R-squared (Within):               0.4893
Date:                Thu, Apr 23 2026   R-squared (Overall):              0.4081
Time:                        23:43:23   Log-likelihood                    95.400
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      504.59
Entities:                          24   P-value                           0.0000
Avg Obs:                       21.000   Distribution:                   F(2,458)
Min Obs:                       21.000                                           
Max Obs:                       21.000   F-statistic (robust):             82.190
                            